In [4]:
# Standard library
import os
import random
import time
from pathlib import Path

# Data
import numpy as np
import pandas as pd

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit-learn
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score
)

# PyTorch
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import (
    Dataset,
    DataLoader
)

# Torchvision
import torchvision
import torchvision.transforms as transforms

# Progress bars
from tqdm import tqdm

# Experiment tracking
import wandb


plt.style.use("default")
import sys
from dataclasses import dataclass

import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from dataclasses import dataclass
from PIL import Image
import sys
import os


In [5]:
# 1. Clone your clean repository code from GitHub
REPO_URL = "https://github.com/K0NSTANT1N3/Facial-Expression-Recognition.git"
REPO_NAME = "Facial-Expression-Recognition"

if not os.path.exists(REPO_NAME):
    print("Cloning repository...")
    os.system(f"git clone {REPO_URL}")
else:
    print("Repository already exists. Pulling latest updates...")
    os.system(f"cd {REPO_NAME} && git pull")

# 2. THE SECRET SAUCE: Add your cloned repo directory to Python's path
sys.path.append(os.path.abspath(REPO_NAME))

print("Successfully linked GitHub repository and adjusted system paths!")


Repository already exists. Pulling latest updates...
Already up to date.
Successfully linked GitHub repository and adjusted system paths!


In [6]:
import wandb

wandb.login(key="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")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


True

In [8]:

@dataclass
class Config:
    csv_path: str = "/kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/train.csv"
    batch_size: int = 64
    lr: float = 5e-4
    epochs: int = 20
    num_classes: int = 7
    image_size: int = 48
    random_seed: int = 67
    dropout: float = 0.2
    model_save_path: str = "best_baseline_model.pt"

config = Config()

EMOTION_LABELS = {0: "Angry", 1: "Disgust", 2: "Fear", 3: "Happy", 4: "Sad", 5: "Surprise", 6: "Neutral"}

def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(config.random_seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

def pixels_to_image(pixel_string):
    pixels = np.fromstring(pixel_string, dtype=np.uint8, sep=" ")
    return pixels.reshape(48, 48)

train_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(10),
    transforms.ToTensor()
])

val_transform = transforms.Compose([
    transforms.ToPILImage(),
    transforms.ToTensor()
])


Using device: cuda


In [9]:

class FERDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.dataframe)
    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        image = pixels_to_image(row["pixels"]).astype(np.uint8)
        if self.transform:
            image = self.transform(image)
        else:
            image = torch.tensor(image.astype(np.float32) / 255.0).unsqueeze(0)
        label = torch.tensor(row["emotion"], dtype=torch.long)
        return image, label

In [10]:

class SimpleCNN(nn.Module):
    def __init__(self, dropout):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(),
            nn.Dropout2d(dropout),
            nn.MaxPool2d(kernel_size=2),
            nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.Dropout2d(dropout),
            nn.MaxPool2d(kernel_size=2),
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.Dropout2d(dropout),
            nn.MaxPool2d(kernel_size=2)
        )
        self.classifier = nn.Sequential(
            nn.Linear(64 * 6 * 6, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Linear(256, 7)
        )
    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, start_dim=1)
        x = self.classifier(x)
        return x


In [11]:
def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item()
        correct += (outputs.argmax(dim=1) == labels).sum().item()
    return running_loss / len(loader), correct / len(loader.dataset)


In [12]:

def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            running_loss += loss.item()
            correct += (outputs.argmax(dim=1) == labels).sum().item()
    return running_loss / len(loader), correct / len(loader.dataset)


In [13]:
def train_model():
    print("Loading dataset...")
    df = pd.read_csv(config.csv_path)
    print(f"Dataset shape: {df.shape}")
    train_df, val_df = train_test_split(df, test_size=0.2, random_state=config.random_seed, stratify=df["emotion"])
    class_counts = train_df["emotion"].value_counts().sort_index()
    class_weights = 1.0 / torch.tensor(class_counts.values, dtype=torch.float32)
    class_weights = (class_weights / class_weights.sum()).to(device)
    train_dataset = FERDataset(train_df, transform=train_transform)
    val_dataset = FERDataset(val_df, transform=val_transform)
    train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True, num_workers=2, pin_memory=True)
    val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False, num_workers=2, pin_memory=True)
    model = SimpleCNN(dropout=config.dropout).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=config.lr)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
    best_val_acc = 0.0
    for epoch in range(config.epochs):
        train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc = validate(model, val_loader, criterion, device)
        print(f"Epoch {epoch+1}/{config.epochs} | Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), config.model_save_path)
            print(f"  -> Saved best model (val_acc: {val_acc:.4f})")
    print(f"\nTraining complete. Best val acc: {best_val_acc:.4f}")
    print("\nRunning final evaluation on validation set...")
    model.load_state_dict(torch.load(config.model_save_path, map_location=device))
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            preds = model(images).argmax(dim=1)
            y_true.extend(labels.numpy())
            y_pred.extend(preds.cpu().numpy())
    print(classification_report(y_true, y_pred, target_names=list(EMOTION_LABELS.values())))
    return model

In [14]:
def load_model(model_path):
    model = SimpleCNN(dropout=config.dropout).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    print(f"Model loaded from {model_path}")
    return model

def predict_from_pixels(model, pixel_string):
    image = pixels_to_image(pixel_string).astype(np.uint8)
    tensor = val_transform(image).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(tensor)
        probs = torch.softmax(logits, dim=1).squeeze()
        pred_idx = probs.argmax().item()
    return pred_idx, EMOTION_LABELS[pred_idx], probs.cpu().numpy()

def predict_from_image_file(model, image_path):
    image = Image.open(image_path).convert("L").resize((48, 48))
    np_image = np.array(image, dtype=np.uint8)
    tensor = val_transform(np_image).unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(tensor)
        probs = torch.softmax(logits, dim=1).squeeze()
        pred_idx = probs.argmax().item()
    return pred_idx, EMOTION_LABELS[pred_idx], probs.cpu().numpy()

def run_inference_on_csv(model, csv_path, num_samples=10):
    df = pd.read_csv(csv_path)
    samples = df.sample(n=num_samples, random_state=config.random_seed).reset_index(drop=True)
    print(f"\nRunning inference on {num_samples} random samples from {csv_path}:\n")
    for i, row in samples.iterrows():
        pred_idx, pred_label, probs = predict_from_pixels(model, row["pixels"])
        true_label = EMOTION_LABELS[row["emotion"]]
        top2 = sorted(enumerate(probs), key=lambda x: x[1], reverse=True)[:2]
        top2_str = ", ".join([f"{EMOTION_LABELS[j]}: {p:.2f}" for j, p in top2])
        match = "✓" if pred_idx == row["emotion"] else "✗"
        print(f"Sample {i+1:>2} | True: {true_label:<10} | Pred: {pred_label:<10} | {match} | Top2: [{top2_str}]")


In [15]:

if __name__ == "__main__":
    if os.path.exists(config.model_save_path):
        print(f"Found existing model at {config.model_save_path}. Loading...")
        model = load_model(config.model_save_path)
    else:
        print("No saved model found. Training from scratch...")
        model = train_model()
    run_inference_on_csv(model, config.csv_path, num_samples=10)
    if len(sys.argv) > 1:
        image_path = sys.argv[1]
        if os.path.exists(image_path):
            pred_idx, pred_label, probs = predict_from_image_file(model, image_path)
            print(f"\nInference on {image_path}:")
            print(f"Predicted emotion: {pred_label} (class {pred_idx})")
            for idx, p in sorted(enumerate(probs), key=lambda x: x[1], reverse=True):
                print(f"  {EMOTION_LABELS[idx]:<10}: {p:.4f}")
        else:
            print(f"File not found: {image_path}")

Found existing model at best_baseline_model.pt. Loading...
Model loaded from best_baseline_model.pt

Running inference on 10 random samples from /kaggle/input/competitions/challenges-in-representation-learning-facial-expression-recognition-challenge/train.csv:

Sample  1 | True: Sad        | Pred: Sad        | ✓ | Top2: [Sad: 0.33, Fear: 0.27]
Sample  2 | True: Sad        | Pred: Angry      | ✗ | Top2: [Angry: 0.29, Fear: 0.23]
Sample  3 | True: Happy      | Pred: Happy      | ✓ | Top2: [Happy: 0.98, Neutral: 0.01]
Sample  4 | True: Sad        | Pred: Sad        | ✓ | Top2: [Sad: 0.45, Fear: 0.24]
Sample  5 | True: Happy      | Pred: Happy      | ✓ | Top2: [Happy: 0.84, Angry: 0.05]
Sample  6 | True: Happy      | Pred: Happy      | ✓ | Top2: [Happy: 0.66, Fear: 0.16]
Sample  7 | True: Fear       | Pred: Surprise   | ✗ | Top2: [Surprise: 0.28, Neutral: 0.25]
Sample  8 | True: Happy      | Pred: Happy      | ✓ | Top2: [Happy: 0.76, Neutral: 0.09]
Sample  9 | True: Fear       | Pred: Fear